# Experiment 6: GloVe Word Embeddings and N-gram Language Model

**Aim:** To implement GloVe word embeddings from scratch, compare them with a pre-trained GloVe model, and build statistical Bigram and Trigram language models for next-word prediction.


## Part 1: GloVe from Scratch

A small self-contained corpus is used to build a vocabulary and weighted word co-occurrence matrix. Simplified GloVe vectors are then learned using gradient descent.


In [1]:
import re
import math
import numpy as np

np.random.seed(42)

corpus_text = """
the king loved the queen and the queen loved the king.
the king and queen ruled the royal kingdom.
the prince is the son of the king and queen.
the queen is a wise royal leader and the king is a strong royal leader.
the prince and princess live in the royal palace.
the kingdom respects the king and the queen.
"""

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

words = tokenize(corpus_text)
vocab = sorted(set(words))
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}
vocab_size = len(vocab)

print("Tokens:", len(words))
print("Vocabulary size:", vocab_size)
print("Vocabulary:", vocab)


Tokens: 61
Vocabulary size: 21
Vocabulary: ['a', 'and', 'in', 'is', 'king', 'kingdom', 'leader', 'live', 'loved', 'of', 'palace', 'prince', 'princess', 'queen', 'respects', 'royal', 'ruled', 'son', 'strong', 'the', 'wise']


In [2]:
window_size = 4
co_occurrence = np.zeros((vocab_size, vocab_size), dtype=np.float64)

for i, center_word in enumerate(words):
    center_id = word_to_id[center_word]
    start = max(0, i - window_size)
    end = min(len(words), i + window_size + 1)

    for j in range(start, end):
        if i == j:
            continue
        context_id = word_to_id[words[j]]
        distance = abs(i - j)
        co_occurrence[center_id, context_id] += 1.0 / distance

print("Co-occurrence matrix shape:", co_occurrence.shape)
print("Non-zero entries:", np.count_nonzero(co_occurrence))


Co-occurrence matrix shape: (21, 21)
Non-zero entries: 207


In [3]:
embedding_dim = 50
epochs = 150
learning_rate = 0.01
x_max = 100
alpha = 0.75

W = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
W_context = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
b = np.zeros(vocab_size)
b_context = np.zeros(vocab_size)

def weight_fn(x):
    return (x / x_max) ** alpha if x < x_max else 1.0

for epoch in range(epochs):
    total_loss = 0.0

    for i in range(vocab_size):
        for j in range(vocab_size):
            x_ij = co_occurrence[i, j]
            if x_ij <= 0:
                continue

            weight = weight_fn(x_ij)
            diff = np.dot(W[i], W_context[j]) + b[i] + b_context[j] - math.log(x_ij)
            total_loss += weight * diff**2

            grad_w_i = 2 * weight * diff * W_context[j].copy()
            grad_w_j = 2 * weight * diff * W[i].copy()
            grad_b = 2 * weight * diff

            W[i] -= learning_rate * grad_w_i
            W_context[j] -= learning_rate * grad_w_j
            b[i] -= learning_rate * grad_b
            b_context[j] -= learning_rate * grad_b

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch + 1:3d}/{epochs} - Loss: {total_loss:.4f}")

embeddings = W + W_context


Epoch  25/150 - Loss: 4.5613
Epoch  50/150 - Loss: 3.5589
Epoch  75/150 - Loss: 3.2086
Epoch 100/150 - Loss: 3.0167
Epoch 125/150 - Loss: 2.8811
Epoch 150/150 - Loss: 2.7756


In [4]:
def find_most_similar_scratch(target_word, top_n=5):
    if target_word not in word_to_id:
        return []

    target_vec = embeddings[word_to_id[target_word]]
    target_norm = np.linalg.norm(target_vec)
    similarities = []

    for i, word in id_to_word.items():
        if word == target_word:
            continue
        vec = embeddings[i]
        denom = target_norm * np.linalg.norm(vec)
        similarity = np.dot(target_vec, vec) / denom if denom else 0.0
        similarities.append((word, float(similarity)))

    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_n]

print("Most similar words to 'king' (GloVe from scratch):")
for word, score in find_most_similar_scratch("king"):
    print(f"{word:12s} {score:.4f}")


Most similar words to 'king' (GloVe from scratch):
the          0.9243
queen        0.6989
and          0.6855
loved        0.1052
is           0.0898


## Part 2: Pre-trained GloVe using Gensim

The standard `glove-wiki-gigaword-100` vectors are downloaded automatically through Gensim. **Internet access is required**, but no manual dataset upload is needed.


In [5]:
!pip -q install gensim


In [7]:
import gensim.downloader as api

print("Loading glove-wiki-gigaword-100...")
glove_model = api.load("glove-wiki-gigaword-100")

word_to_query = "king"
print(f"\nMost similar words to '{word_to_query}' using pre-trained GloVe:")
for word, similarity in glove_model.most_similar(word_to_query, topn=5):
    print(f"{word:12s} {similarity:.4f}")


Loading glove-wiki-gigaword-100...

Most similar words to 'king' using pre-trained GloVe:
prince       0.7682
queen        0.7508
son          0.7021
brother      0.6986
monarch      0.6978


## Part 3: Statistical N-gram Language Model from Scratch

Bigram and Trigram models are trained on a small built-in NLP corpus. Maximum-likelihood probabilities are used for next-word prediction.


In [8]:
from collections import defaultdict, Counter

ngram_corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

class NGramLanguageModel:
    def __init__(self, n=2):
        self.n = n
        self.ngrams = defaultdict(Counter)
        self.context_counts = defaultdict(int)

    @staticmethod
    def tokenize(text):
        return re.findall(r"[a-z]+|[.]", text.lower())

    def fit(self, corpus):
        for sentence in corpus:
            tokens = ["<s>"] * (self.n - 1) + self.tokenize(sentence) + ["</s>"]
            for i in range(len(tokens) - self.n + 1):
                context = tuple(tokens[i:i + self.n - 1])
                target = tokens[i + self.n - 1]
                self.ngrams[context][target] += 1
                self.context_counts[context] += 1

    def get_next_word_probabilities(self, context_tokens):
        size = self.n - 1
        context_tokens = [w.lower() for w in context_tokens]
        if len(context_tokens) < size:
            context_tokens = ["<s>"] * (size - len(context_tokens)) + context_tokens
        else:
            context_tokens = context_tokens[-size:]

        context = tuple(context_tokens)
        total = self.context_counts[context]
        if total == 0:
            return {}

        probs = {word: count / total for word, count in self.ngrams[context].items()}
        return dict(sorted(probs.items(), key=lambda x: x[1], reverse=True))

    def predict_next_word(self, context_tokens):
        probs = self.get_next_word_probabilities(context_tokens)
        return next(iter(probs.items())) if probs else (None, 0.0)

bigram_model = NGramLanguageModel(n=2)
trigram_model = NGramLanguageModel(n=3)
bigram_model.fit(ngram_corpus)
trigram_model.fit(ngram_corpus)


In [9]:
print("--- Bigram Next-Word Predictions ---")
for context in [["natural"], ["machine"], ["artificial"], ["language"]]:
    pred, prob = bigram_model.predict_next_word(context)
    print(f"Context {context} -> {pred!r}, probability={prob:.4f}")
    print("Options:", bigram_model.get_next_word_probabilities(context))

print("\n--- Trigram Next-Word Predictions ---")
for context in [["natural", "language"], ["machine", "learning"], ["artificial", "intelligence"], ["language", "model"]]:
    pred, prob = trigram_model.predict_next_word(context)
    print(f"Context {context} -> {pred!r}, probability={prob:.4f}")
    print("Options:", trigram_model.get_next_word_probabilities(context))


--- Bigram Next-Word Predictions ---
Context ['natural'] -> 'language', probability=1.0000
Options: {'language': 1.0}
Context ['machine'] -> 'learning', probability=1.0000
Options: {'learning': 1.0}
Context ['artificial'] -> 'intelligence', probability=1.0000
Options: {'intelligence': 1.0}
Context ['language'] -> 'processing', probability=0.4286
Options: {'processing': 0.42857142857142855, 'models': 0.2857142857142857, '.': 0.14285714285714285, 'model': 0.14285714285714285}

--- Trigram Next-Word Predictions ---
Context ['natural', 'language'] -> 'processing', probability=1.0000
Options: {'processing': 1.0}
Context ['machine', 'learning'] -> 'is', probability=0.3333
Options: {'is': 0.3333333333333333, '.': 0.3333333333333333, 'and': 0.3333333333333333}
Context ['artificial', 'intelligence'] -> '.', probability=0.6667
Options: {'.': 0.6666666666666666, 'includes': 0.3333333333333333}
Context ['language', 'model'] -> 'predicts', probability=1.0000
Options: {'predicts': 1.0}


## Conclusion

This experiment demonstrates two approaches to language representation and prediction: GloVe learns dense semantic word vectors from co-occurrence statistics, while N-gram models estimate the probability of the next word from local word histories. Pre-trained GloVe generally provides stronger semantic relationships because it was trained on a much larger corpus.
